# Dynamic JSCC on Kaggle — train and/or generate figures

Runs the model straight from
[`quantum-dynamic-jscc`](https://github.com/rowshan-mannan-oni/quantum-dynamic-jscc). The notebook holds no copy of the model: it clones the
repository and calls its scripts, so everything reflects the latest commit.

Set `MODE` in the config cell below:

| `MODE` | What happens |
|---|---|
| `"train"` | Train from scratch here, then plot the figures from the result |
| `"weights"` | Skip training and use a checkpoint you attached as a Kaggle Dataset |

**Kaggle settings:** Accelerator -> **GPU**, Internet -> **On** (needed to clone the repo).

**Runtime warning.** The paper schedule is 400 epochs and takes roughly **2–4 h** on a T4/P100.
Kaggle stops a session well before that if you leave it idle, and the repository has **no
working resume** — the learning-rate stage transitions test `epoch == ...` exactly and the
Gumbel temperature is not checkpointed, so a restarted run silently continues on the wrong
schedule. Either let one run finish in a single session, or start with `QUICK_TEST = True` to
verify the pipeline first.

## 1. Configuration — the only cell you normally edit

In [ ]:
# ---------------- what to do ----------------
MODE = "train"           # "train"  -> train here, then make figures
                         # "weights" -> use a checkpoint from /kaggle/input

QUICK_TEST = True        # True: a few epochs just to prove the pipeline runs.
                         # Set False for the real 400-epoch paper schedule.

# ------------- model / run settings -------------
# These four also determine the checkpoint folder name, so training and the
# figure step must agree on them.
LAMBDA_REWARD = 1.5e-3   # rate penalty; sweep this to trace the tradeoff curve
SELECT        = "hard"   # "hard" or "soft" mask selection
C_CHANNEL     = 16
LAMBDA_L2     = 1

# ---------------- training ----------------
BATCH_SIZE      = 128    # 128 is the paper setting
SNR_MIN, SNR_MAX = 0, 20
NUM_WORKERS     = 2      # Kaggle is Linux, so workers are safe here
SAVE_EPOCH_FREQ = 20     # save a numbered checkpoint this often

if QUICK_TEST:
    N_JOINT, N_DECAY, N_FINE = 3, 3, 2
else:
    N_JOINT, N_DECAY, N_FINE = 150, 150, 100      # paper schedule

# ---------------- evaluation ----------------
NUM_TEST = 10000                                   # lower for a fast pass
SNR_LIST = "0,2,4,6,8,10,12,14,16,18,20"
FIG6_SNR = 10

TOTAL_EPOCHS = N_JOINT + N_DECAY + N_FINE
CKPT_NAME = f"C{C_CHANNEL}_L2_{LAMBDA_L2}_re_{LAMBDA_REWARD}_{SELECT}"
print(f"MODE={MODE} | epochs={TOTAL_EPOCHS} | checkpoint folder: {CKPT_NAME}")
if MODE == "train" and not QUICK_TEST:
    print("\nFull schedule selected: expect roughly 2-4 h on a T4/P100.")

## 2. Clone (or update) the repository

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/rowshan-mannan-oni/quantum-dynamic-jscc.git"
REPO_DIR = "/kaggle/working/quantum-dynamic-jscc"

if os.path.isdir(REPO_DIR):
    print("Repository present - pulling latest commit")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("\nWorking directory:", os.getcwd())
subprocess.run(["git", "log", "-1", "--format=Using commit %h - %s"], check=True)

## 3. Dependencies

Kaggle already ships a CUDA build of PyTorch. **Do not** install the pinned torch from
`requirements.txt` here — that would replace a working GPU install. Only the plotting and
metric packages are needed.

In [ ]:
import importlib, subprocess, sys

for module, package in [("skimage", "scikit-image"), ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(module) is None:
        print(f"installing {package} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
    else:
        print(f"{package} already available")

import torch
print("\ntorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Enable it under Settings -> Accelerator.")

GPU_IDS = "0" if torch.cuda.is_available() else "-1"

## 4. CIFAR-10

Uses an attached Kaggle CIFAR-10 dataset if present, otherwise downloads it. The scripts read
from `./data` inside the repository, so everything is staged there.

In [ ]:
import glob, os, shutil, tarfile

DATA_ROOT = os.path.join(REPO_DIR, "data")
os.makedirs(DATA_ROOT, exist_ok=True)
target = os.path.join(DATA_ROOT, "cifar-10-batches-py")

def find(pattern):
    hits = glob.glob(pattern, recursive=True)
    return hits[0] if hits else None

if not os.path.isdir(target):
    folder = find("/kaggle/input/**/cifar-10-batches-py")
    tar = find("/kaggle/input/**/cifar-10-python.tar.gz")
    if folder:
        print("Found extracted dataset:", folder)
        shutil.copytree(folder, target)
    elif tar:
        print("Found tarball:", tar)
        with tarfile.open(tar) as t:
            t.extractall(DATA_ROOT)
    else:
        print("No CIFAR-10 under /kaggle/input - it will download (Internet must be On).")

print("Dataset ready:", os.path.isdir(target))

## 5a. Train  *(runs only when `MODE == "train"`)*

Calls the repository's `train_dyna.py`. Output is streamed live so you can watch the loss.
Checkpoints are written to `Checkpoints/<CKPT_NAME>/`, which is exactly where the figure step
looks for them — no conversion needed.

In [ ]:
import subprocess, sys, time

if MODE == "train":
    cmd = [sys.executable, "-u", "train_dyna.py",
           "--gpu_ids", GPU_IDS,
           "--select", SELECT,
           "--C_channel", str(C_CHANNEL),
           "--lambda_L2", str(LAMBDA_L2),
           "--lambda_reward", str(LAMBDA_REWARD),
           "--batch_size", str(BATCH_SIZE),
           "--SNR_MIN", str(SNR_MIN), "--SNR_MAX", str(SNR_MAX),
           "--n_epochs_joint", str(N_JOINT),
           "--n_epochs_decay", str(N_DECAY),
           "--n_epochs_fine", str(N_FINE),
           "--num_workers", str(NUM_WORKERS),
           "--save_epoch_freq", str(SAVE_EPOCH_FREQ),
           "--print_freq", "12800"]
    print(" ".join(cmd), "\n")
    start = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"training failed with exit code {proc.returncode}")
    print(f"\nTraining finished in {(time.time()-start)/60:.1f} min")
else:
    print('MODE is not "train" - skipping.')

### Training curves

`train_dyna.py` logs to `Checkpoints/<name>/loss_log.txt`; this parses that file and plots it.
The dashed lines mark the learning-rate stage boundaries.

In [ ]:
import os, re
import matplotlib.pyplot as plt
import numpy as np

log_path = os.path.join("Checkpoints", CKPT_NAME, "loss_log.txt")
if os.path.exists(log_path):
    pat = re.compile(r"epoch:\s*(\d+).*?G_L2:\s*([\d.eE+-]+)\s+G_reward:\s*([\d.eE+-]+)")
    rows = [(int(m.group(1)), float(m.group(2)), float(m.group(3)))
            for m in (pat.search(l) for l in open(log_path)) if m]
    if rows:
        arr = np.array(rows)
        epochs = np.unique(arr[:, 0])
        l2 = [arr[arr[:, 0] == e, 1].mean() for e in epochs]
        rw = [arr[arr[:, 0] == e, 2].mean() for e in epochs]
        psnr = [10 * np.log10(4.0 / max(v, 1e-12)) for v in l2]     # images in [-1,1]

        fig, ax = plt.subplots(1, 3, figsize=(15, 4))
        for a, y, t, yl in [(ax[0], l2, "Reconstruction MSE", "MSE"),
                            (ax[1], psnr, "Train PSNR (approx)", "PSNR (dB)"),
                            (ax[2], rw, "Avg active selective groups", "# groups")]:
            a.plot(epochs, y); a.set_title(t); a.set_xlabel("epoch"); a.set_ylabel(yl)
            a.grid(alpha=.3)
            for b in (N_JOINT, N_JOINT + N_DECAY):
                if epochs.max() > b:
                    a.axvline(b, color="gray", ls="--", lw=.8)
        plt.tight_layout(); plt.show()
    else:
        print("No parsable lines in", log_path)
else:
    print("No training log at", log_path, "- expected when MODE is 'weights'.")

## 5b. Use existing weights  *(runs only when `MODE == "weights"`)*

Finds a checkpoint under `/kaggle/input` that packs all four sub-networks as
`{'SE','CE','G','P'}` and converts it into the per-network layout the repository expects.
Attach it via **Add Input** — weights are not stored in the repository.

`convert_kaggle_weights.py` validates every state dict with `strict=True` before writing, so a
mismatched architecture fails loudly rather than silently.

In [ ]:
import glob, subprocess, sys, torch

if MODE == "weights":
    def looks_like_checkpoint(path):
        try:
            ck = torch.load(path, map_location="cpu")
            return isinstance(ck, dict) and {"SE", "CE", "G", "P"}.issubset(ck.keys())
        except Exception:
            return False

    candidates = sorted(glob.glob("/kaggle/input/**/*.pth", recursive=True))
    weights = next((p for p in candidates if looks_like_checkpoint(p)), None)

    if weights is None:
        raise SystemExit(
            "No checkpoint containing {'SE','CE','G','P'} found under /kaggle/input.\n"
            "Upload your final.pth as a Kaggle Dataset and attach it with Add Input.\n"
            f"Files scanned: {candidates}")

    print("Using weights:", weights)
    subprocess.run([sys.executable, "convert_kaggle_weights.py",
                    "--src", weights,
                    "--lambda_reward", str(LAMBDA_REWARD),
                    "--lambda_L2", str(LAMBDA_L2),
                    "--select", SELECT,
                    "--C_channel", str(C_CHANNEL),
                    "--force"], check=True)
else:
    print('MODE is not "weights" - skipping.')

## 6. Check a checkpoint is in place

In [ ]:
import os

ckpt_dir = os.path.join("Checkpoints", CKPT_NAME)
expected = [f"latest_net_{n}.pth" for n in ("SE", "CE", "G", "P")]
missing = [f for f in expected if not os.path.exists(os.path.join(ckpt_dir, f))]

if missing:
    raise SystemExit(f"Missing {missing} in {ckpt_dir}.\n"
                     "Run the training cell, or set MODE='weights' and attach a checkpoint.")

for f in expected:
    p = os.path.join(ckpt_dir, f)
    print(f"  {f:22s} {os.path.getsize(p)/1e6:6.2f} MB")
print("\nCheckpoint ready in", ckpt_dir)

## 7. Generate the figures

`make_figures.py` runs the SNR sweep, the fixed-rate sweep, the per-class breakdown and sample
reconstructions, writing PNGs plus the CSVs behind them.

In [ ]:
import subprocess, sys

FIG_DIR = "/kaggle/working/figures"

cmd = [sys.executable, "-u", "make_figures.py",
       "--gpu_ids", GPU_IDS,
       "--select", SELECT,
       "--C_channel", str(C_CHANNEL),
       "--lambda_L2", str(LAMBDA_L2),
       "--lambda_reward", str(LAMBDA_REWARD),
       "--num_test", str(NUM_TEST),
       "--snr_list", SNR_LIST,
       "--fig6_snr", str(FIG6_SNR),
       "--num_workers", str(NUM_WORKERS),
       "--dataroot", DATA_ROOT,
       "--results_dir", FIG_DIR]
print(" ".join(cmd), "\n")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"make_figures.py failed with exit code {proc.returncode}")

## 8. Show the figures

In [ ]:
from IPython.display import Image, display, Markdown
import os

titles = {
    "fig4_rate_psnr_vs_snr.png": "Average rate (CPP) and PSNR vs SNR",
    "fig5_attainable_psnr.png":  "Attainable PSNR: fixed rates vs adaptive",
    "fig6_per_class.png":        "Per-class rate and PSNR",
    "reconstructions.png":       "Sample reconstructions",
}
for name, title in titles.items():
    path = os.path.join(FIG_DIR, name)
    if os.path.exists(path):
        display(Markdown(f"### {title}"))
        display(Image(filename=path))
    else:
        print("missing:", path)

In [ ]:
import pandas as pd, os

for csv_name in ["fig4_data.csv", "fig5_data.csv", "fig6_data.csv"]:
    path = os.path.join(FIG_DIR, csv_name)
    if os.path.exists(path):
        print(f"\n===== {csv_name} =====")
        display(pd.read_csv(path))

## 9. Scalar metrics *(optional)*

`test_dyna.py` prints PSNR, SSIM and per-class rates at a single SNR. It evaluates one image at
a time, so keep `--num_test` modest.

In [ ]:
import subprocess, sys

TEST_SNR = 10
subprocess.run([sys.executable, "-u", "test_dyna.py",
                "--gpu_ids", GPU_IDS,
                "--select", SELECT,
                "--C_channel", str(C_CHANNEL),
                "--lambda_L2", str(LAMBDA_L2),
                "--lambda_reward", str(LAMBDA_REWARD),
                "--SNR", str(TEST_SNR),
                "--num_test", "2000",
                "--num_test_channel", "1",
                "--num_workers", str(NUM_WORKERS)], check=True)

## Notes

- Figures, CSVs and checkpoints live under `/kaggle/working` and can be downloaded from the
  notebook's **Output** tab. Save the checkpoint if you want to reuse it — the working
  directory does not survive the session.
- Re-running cell 2 pulls the newest commit. Restart the kernel after a pull if model code
  changed, so Python re-imports it.
- **Resume is not supported.** The stage transitions in `train_dyna.py` use exact epoch
  equality and the Gumbel temperature is not saved, so `--continue_train` silently resumes on
  the wrong learning rate and temperature. Treat each run as one session.
- To trace the rate–distortion curve, train several times with different `LAMBDA_REWARD`
  values; each produces one operating point and its own checkpoint folder.
- Set `QUICK_TEST = True` first. It runs 8 epochs, which is enough to confirm the whole
  pipeline works before committing to a multi-hour run.